### Import des librairies utiles pour ce projet

In [2141]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import statsmodels.api as sm

### Chargement du dataset selon les csv remis lors de la mission

In [2142]:
df_lapage=pd.read_csv(r'..\..\projet9_Lapage\data\processed\df_lapage.csv',sep=';', encoding='utf-8')

### Demandes d'Annabelle
 
Nous souhaitons élaborer différents graphiques autour du chiffre d'affaires comme par exemples l’évolution dans le temps du :

- chiffre d’affaires avec la moyenne mobile (tu pourras choisir la période : jour, semaine, mois, etc.),
- chiffre d’affaires par catégorie,
- nombre de clients par mois,
- nombre de transactions,
- nombre de produits vendus,
- etc.

Il serait également intéressant de faire un zoom sur les références :
- les tops,
- les flops,
- la répartition par catégorie,
- etc.

Enfin, j’aimerais avoir quelques informations sur les profils de nos clients :
- répartition du chiffre d'affaires pour les clients BtoB,
- courbe de Lorenz,
- etc.

Après, toutes les informations et tous graphiques qui apporteraient de
l’information pertinente sont les bienvenus !

In [2143]:
print('Les informations minimales par colonnes du dataframe sont les suivantes :')
print(df_lapage.min())

print('\n ----------------------------------------------------------')
print('Les informations maximales par colonnes du dataframe sont les suivantes :')
print(df_lapage.max())

print('\n ----------------------------------------------------------')
print('Le type de chaque colonne du dataframe est :')
print(df_lapage.dtypes)

Les informations minimales par colonnes du dataframe sont les suivantes :
id_prod              0_0
date          2021-03-01
session_id           s_1
client_id            c_1
price               0.62
categ                  0
sex                    f
birth               1929
dtype: object

 ----------------------------------------------------------
Les informations maximales par colonnes du dataframe sont les suivantes :
id_prod             2_99
date          2023-02-28
session_id       s_99999
client_id          c_999
price              300.0
categ                  2
sex                    m
birth               2004
dtype: object

 ----------------------------------------------------------
Le type de chaque colonne du dataframe est :
id_prod        object
date           object
session_id     object
client_id      object
price         float64
categ           int64
sex            object
birth           int64
dtype: object


In [2144]:
# Conversion de la colonen date en datetime
df_lapage['date'] = pd.to_datetime(df_lapage['date'])

#### Chiffre d’affaires avec la moyenne mobile : jour, semaine, mois

In [2145]:
# Agregation en jour pour ensuite effectuer la moyenne glissante en semaine 
df_ca = df_lapage.groupby(pd.Grouper(key='date', freq='D'))['price'].sum().to_frame().reset_index()

In [2146]:
# Création de la colonne indiquant une moyenne glissante sur une semaine
df_ca['Moyenne glissante (semaine)']=df_ca['price'].rolling(window=7).mean().round(2)

In [2147]:
# Création de la colonne indiquant une moyenne glissante sur un mois
df_ca['Moyenne glissante (mois)']=df_ca['price'].rolling(window=30).mean().round(2)

In [2148]:
# Vérification de la création des colonnes de moyennes glissantes
df_ca.head(35)

,date,price,Moyenne glissante (semaine),Moyenne glissante (mois)
0,2021-03-01,16565.22,NaN,NaN
1,2021-03-02,15486.45,NaN,NaN
2,2021-03-03,15198.69,NaN,NaN
3,2021-03-04,15196.07,NaN,NaN
4,2021-03-05,17471.37,NaN,NaN
5,2021-03-06,15785.28,NaN,NaN
6,2021-03-07,14760.20,15780.47,NaN
7,2021-03-08,15679.53,15653.94,NaN
8,2021-03-09,15710.51,15685.95,NaN
9,2021-03-10,15496.87,15728.55,NaN


In [2149]:
# Grahique concernant la saisonnaluté du CA. Pas de réelle saisonalité
# Interprétation limitée et tendance à une certaine stabilité - saisonnalité marquée mais pas de pics

figure = px.line(
    df_ca, 
    x='date', 
    y=['price', 'Moyenne glissante (semaine)','Moyenne glissante (mois)'], 
    color_discrete_sequence=px.colors.qualitative.Pastel,
    markers=False,
    title='Évolution du chiffre d\'affaires',
    labels={
        'date': 'Date (mois)',
        'value': 'Chiffre d’affaires (€)',
        'variable': 'Moyennes glissantes'
    }
)

figure.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure.update_traces(
    hovertemplate='%{y:.0f}€'
)

figure.show()

#### Chiffre d’affaires par catégorie + évolution dans le temps

In [2150]:
# Création d'un df contenant les données agregées par mois
df_lapage_mois = df_lapage.groupby([pd.Grouper(key='date', freq='MS'), 'categ'])['price'].sum().reset_index()
df_lapage_mois['date AAAA-MM'] = df_lapage_mois['date'].dt.strftime('%Y-%m')

In [2151]:
# Vérification des catégories uniques présentes dans la colonne categ
categories =df_lapage_mois.categ.unique()
print('Les catégories uniques du df sont les suivantes :', categories)

Les catégories uniques du df sont les suivantes : [0 1 2]


In [2152]:
# Affichage du chiffre d'affaire par catégorie sur l'intégralité de la période du dataset
for categ in df_lapage_mois.categ.unique():
    ca = df_lapage_mois[df_lapage_mois['categ'] == categ]['price'].sum()
    print (f'Le chiffre d\'affaire total de la catégorie {categ} est de {ca:,.0f} euros'.replace(',',' '))

Le chiffre d'affaire total de la catégorie 0 est de 4 419 731 euros
Le chiffre d'affaire total de la catégorie 1 est de 4 827 657 euros
Le chiffre d'affaire total de la catégorie 2 est de 2 780 275 euros


In [2153]:
figure2 = px.line(
    df_lapage_mois, 
    x='date', 
    y='price', 
    color='categ', 
    markers=True,
    title='Évolution mensuelle du chiffre d’affaires par catégorie',
    labels={
        'date': 'Date (mois)',
        'price': 'Chiffre d’affaires (€)',
        'categ': 'Catégories'
    }
)

figure2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure2.update_traces(
    hovertemplate='%{y:.0f}€'
    )


figure2.show()

#### Nombre de clients par mois + évolution dans le temps

In [2154]:
#Affichage du nombre de clients au total sur la période du dataset
print(f'Sur l\'intégralité de la période de référence du dataset, il y a eu un total de {df_lapage['client_id'].nunique()} clients distincts.')

Sur l'intégralité de la période de référence du dataset, il y a eu un total de 8600 clients distincts.


In [2155]:
#Affichage du nombre de clients uniques par mois
df_clients_par_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].nunique().reset_index(name='Nombre de clients'))
df_clients_par_mois.head()

,date,Nombre de clients
0,2021-03-01,5676
1,2021-04-01,5674
2,2021-05-01,5644
3,2021-06-01,5659
4,2021-07-01,5672


In [2156]:
figure3=px.line(
    df_clients_par_mois,
    x='date',
    y='Nombre de clients',
    markers=True,
    title='Évolution mensuelle du nombre de clients (uniques)',
    labels={
        'date': 'Date (mois)'
    }
)

figure3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure3.show()

In [2157]:
#Affichage du nombre de clients par mois
df_clients_total_par_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].count().reset_index(name='Nombre de clients'))
df_clients_total_par_mois.head()

,date,Nombre de clients
0,2021-03-01,28601
1,2021-04-01,28443
2,2021-05-01,28285
3,2021-06-01,26850
4,2021-07-01,24738


In [2158]:
figure4=px.line(
    df_clients_total_par_mois,
    x='date',
    y='Nombre de clients',
    markers=True,
    title='Évolution mensuelle du nombre de clients',
    labels={
        'date': 'Date (mois)'
    }
)

figure4.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure4.show()

#### Nombre de transactions + évolution dans le temps

In [2159]:
#Affichage du nombre de transactions totales sur la période du dataset
print(f'Sur l\'intégralité de la période de référence du dataset, il y a eu un total de {df_lapage['session_id'].count()} transactions.')

Sur l'intégralité de la période de référence du dataset, il y a eu un total de 687534 transactions.


In [2160]:
#Affichage du nombre de transactions par semaine
df_transactions_semaine = (df_lapage.groupby(pd.Grouper(key='date', freq='W'))['session_id'].count().reset_index(name='Nombre de transactions par semaine'))
df_transactions_semaine.head()

,date,Nombre de transactions par semaine
0,2021-03-07,6524
1,2021-03-14,6422
2,2021-03-21,6590
3,2021-03-28,6368
4,2021-04-04,6406


In [2161]:
#Affichage du nombre de transactions par mois
df_transactions_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['session_id'].count().reset_index(name='Nombre de transactions par mois'))
df_transactions_mois.head()

,date,Nombre de transactions par mois
0,2021-03-01,28601
1,2021-04-01,28443
2,2021-05-01,28285
3,2021-06-01,26850
4,2021-07-01,24738


In [2162]:
#Affichage du nombre de transactions par trimestre
df_transactions_trimestre = (df_lapage.groupby(pd.Grouper(key='date', freq='QS'))['session_id'].count().reset_index(name='Nombre de transactions par trimestre'))
df_transactions_trimestre.head()

### Prendre en compte le fait que la période commence le 01/03/21 donc le premier trimestre n'est pas significatif
### Le dernier trimestre de la période du dataset ne sera pas significatif non plus car il ne couvre que janvier et février 2023

,date,Nombre de transactions par trimestre
0,2021-01-01,28601
1,2021-04-01,83578
2,2021-07-01,83702
3,2021-10-01,90790
4,2022-01-01,88633


In [2163]:
#Réunion des dataframes en un seul en vue de les afficher sur le même graphique
df_transactions= df_transactions_semaine.merge(df_transactions_mois, on = 'date', how='outer').sort_values('date')
df_transactions= df_transactions.merge(df_transactions_trimestre, on = 'date', how='outer').sort_values('date')
df_transactions['Nombre de transactions par mois'] = (df_transactions['Nombre de transactions par mois'].interpolate(method='linear'))
df_transactions['Nombre de transactions par trimestre'] = (df_transactions['Nombre de transactions par trimestre'].interpolate(method='linear'))
df_transactions.head()

,date,Nombre de transactions par semaine,Nombre de transactions par mois,Nombre de transactions par trimestre
0,2021-01-01,NaN,NaN,28601.000000
1,2021-03-01,NaN,28601.0,37763.833333
2,2021-03-07,6524.0,28569.4,46926.666667
3,2021-03-14,6422.0,28537.8,56089.500000
4,2021-03-21,6590.0,28506.2,65252.333333


In [2164]:
figure5=px.line(
    df_transactions,
    x='date',
    y=[ 'Nombre de transactions par semaine','Nombre de transactions par mois','Nombre de transactions par trimestre'],
    markers=False,
    title='Évolution du nombre de transactions par semaine, mois et trimestre',
    labels={
        'date': 'Date (mois)'
    }
)

figure5.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure5.show()

#### Nombre de produits vendus + évolution dans le temps

In [2165]:
# tous les produits sur toute la période, puis par mois, puis par semaine
#Affichage du nombre de produits vendus par semaine
df_lapage_prod_semaine = (df_lapage.groupby(pd.Grouper(key='date', freq='W'))['id_prod'].nunique().reset_index(name='Nombre de produits vendus par semaine'))
df_lapage_prod_semaine.head()

,date,Nombre de produits vendus par semaine
0,2021-03-07,1608
1,2021-03-14,1614
2,2021-03-21,1638
3,2021-03-28,1624
4,2021-04-04,1623


In [2166]:
#Affichage du nombre de produits vendus par mois
df_lapage_prod_mois = (df_lapage.groupby(pd.Grouper(key='date', freq='MS'))['id_prod'].nunique().reset_index(name='Nombre de produits vendus par mois'))

In [2167]:
#Affichage du nombre de produits vendus par trimestre
df_lapage_prod_trimestre = (df_lapage.groupby(pd.Grouper(key='date', freq='QS'))['id_prod'].nunique().reset_index(name='Nombre de produits vendus par trimestre'))

In [2168]:
#Réunion des dataframes en un seul en vue de les afficher sur le même graphique
df_lapage_prod= df_lapage_prod_semaine.merge(df_lapage_prod_mois, on = 'date', how='outer').sort_values('date')
df_lapage_prod= df_lapage_prod.merge(df_lapage_prod_trimestre, on = 'date', how='outer').sort_values('date')
df_lapage_prod['Nombre de produits vendus par mois'] = (df_lapage_prod['Nombre de produits vendus par mois'].interpolate(method='linear'))
df_lapage_prod['Nombre de produits vendus par trimestre'] = (df_lapage_prod['Nombre de produits vendus par trimestre'].interpolate(method='linear'))
df_lapage_prod.head()

,date,Nombre de produits vendus par semaine,Nombre de produits vendus par mois,Nombre de produits vendus par trimestre
0,2021-01-01,NaN,NaN,2482.000000
1,2021-03-01,NaN,2482.0,2563.666667
2,2021-03-07,1608.0,2484.0,2645.333333
3,2021-03-14,1614.0,2486.0,2727.000000
4,2021-03-21,1638.0,2488.0,2808.666667


In [2169]:
figure6=px.line(
    df_lapage_prod,
    x='date',
    y=[ 'Nombre de produits vendus par semaine','Nombre de produits vendus par mois','Nombre de produits vendus par trimestre'],
    markers=False,
    title='Évolution du nombre de produits uniques vendus par semaine, mois et trimestre',
    labels={
        'date': 'Date (mois)'
    }
)

figure6.update_layout(
    template='plotly_white',
    hovermode='x unified',
    width=1050,
    height=500
)

figure6.update_traces(
    hovertemplate='%{y:.0f}€'
    )

figure6.show()

# Top, Flop, la répartition par catégorie

- les tops,
- les flops,
- la répartition par catégorie

In [2170]:
# Création d'une colonne dans le df principal, indiquant la quantité de produits vendus, par article
df_lapage['quantity'] = df_lapage.groupby('id_prod')['id_prod'].transform('count')
df_lapage.head()

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity
0,0_1259,2021-03-01,s_1,c_329,11.99,0,f,1967,341
1,0_1390,2021-03-01,s_2,c_664,19.37,0,m,1960,880
2,0_1352,2021-03-01,s_3,c_580,4.50,0,m,1988,1004
3,0_1458,2021-03-01,s_4,c_7912,6.55,0,f,1989,1065
4,0_1358,2021-03-01,s_5,c_2033,16.49,0,f,1956,1106


In [2171]:
# Création d'un dataframe restreint contenant la colonne quantité
df_quantity = df_lapage[['id_prod', 'price', 'categ', 'quantity']]
df_quantity.head()

,id_prod,price,categ,quantity
0,0_1259,11.99,0,341
1,0_1390,19.37,0,880
2,0_1352,4.50,0,1004
3,0_1458,6.55,0,1065
4,0_1358,16.49,0,1106


In [2172]:
# Eviter le warning concernant les bugs de pandas sur les copies de dataframes
df_quantity = df_quantity.copy()

df_quantity['ca_par_produit'] = df_quantity['quantity'] * df_quantity['price']

In [2173]:
# Les tops par catégorie
for categ in df_quantity['categ'].unique():
    top10=df_quantity[df_quantity['categ'] == categ].sort_values('ca_par_produit', ascending=False).head(10)
    print('\n--------------------------------------------------------------------------------------------- ')
    print(f'Le top 10 des produits les plus vendus de la catégorie {categ} est le suivant :')
    print('---------------------------------------------------------------------------------------------\n')
    print(top10[['id_prod', 'categ','quantity', 'price', 'ca_par_produit']])



--------------------------------------------------------------------------------------------- 
Le top 10 des produits les plus vendus de la catégorie 0 est le suivant :
---------------------------------------------------------------------------------------------

       id_prod  categ  quantity  price  ca_par_produit
667486  0_1441      0      1235  18.99        23452.65
666021  0_1441      0      1235  18.99        23452.65
669747  0_1441      0      1235  18.99        23452.65
163240  0_1441      0      1235  18.99        23452.65
168178  0_1441      0      1235  18.99        23452.65
167771  0_1441      0      1235  18.99        23452.65
167423  0_1441      0      1235  18.99        23452.65
167115  0_1441      0      1235  18.99        23452.65
687249  0_1441      0      1235  18.99        23452.65
166639  0_1441      0      1235  18.99        23452.65

--------------------------------------------------------------------------------------------- 
Le top 10 des produits les plus ve

In [2174]:
# Graphique du top 10 

In [2175]:
# Les flops par catégorie
for categ in df_quantity['categ'].unique():
    top10=df_quantity[df_quantity['categ'] == categ].sort_values('ca_par_produit', ascending=True).head(10)
    print(f'Le top 10 des produits les moins vendus de la catégorie {categ} est le suivant :')
    print(top10[['id_prod', 'categ','quantity', 'price', 'ca_par_produit']])


Le top 10 des produits les moins vendus de la catégorie 0 est le suivant :
       id_prod  categ  quantity  price  ca_par_produit
85656   0_1539      0         1   0.99            0.99
41359   0_1284      0         1   1.38            1.38
293680  0_1653      0         2   0.99            1.98
497028  0_1653      0         2   0.99            1.98
131286   0_807      0         1   1.99            1.99
7437     0_541      0         1   1.99            1.99
6346    0_1601      0         1   1.99            1.99
46079   0_1728      0         1   2.27            2.27
335381  0_1498      0         1   2.48            2.48
266083   0_898      0         2   1.27            2.54
Le top 10 des produits les moins vendus de la catégorie 1 est le suivant :
       id_prod  categ  quantity  price  ca_par_produit
291939   1_420      1         2   7.12           14.24
280711   1_420      1         2   7.12           14.24
262495   1_224      1         4   4.95           19.80
612294   1_224      1    

In [2176]:
#### Utiliser le subplot de plotly

#### Profils de nos clients :
- répartition du chiffre d'affaires pour les clients BtoB,
- courbe de Lorenz,
- etc.


# IDENTIFICATION CLIENTS B TO B ET B TO C

In [2177]:
# Il existe plusieurs moyens pour identifier les B to B : 

In [ ]:
# # CA très élevé et dans ce cas nous identifions 4 clients ayant généré plusieurs centaines de milliers d'euros de CA
ca_par_client = df_lapage.groupby(['client_id'])['price'].sum().reset_index().sort_values('price', ascending=False)
ca_par_client.head(10)
# clients = ['c_1609','c_4958','c_6714','c_3454']

,client_id,price
677,c_1609,326039.89
4388,c_4958,290227.03
6337,c_6714,153918.60
2724,c_3454,114110.57
634,c_1570,5285.82
2513,c_3263,5276.87
1268,c_2140,5260.18
2108,c_2899,5214.05
7006,c_7319,5155.77
7715,c_7959,5135.75


In [ ]:
# Indication de la mention B2B ou B2C selon le tri ci-dessus
clients = ['c_1609','c_4958','c_6714','c_3454']
df_lapage['type_client']=df_lapage['client_id'].apply(lambda x: 'B to B' if x in clients else 'B to C')
df_lapage.head(20)

# Création d'un df b2b et conservation du df principal excluant ces 4 clients
df_b2b = df_lapage[df_lapage['client_id'].isin(clients)]

# Création d'un df b2b et conservation du df principal excluant ces 4 clients
df_b2c = df_lapage[df_lapage['type_client'] == 'B to C']
df_b2c

In [ ]:
# # Le fait de trier les B2B sur la quantité n'est pas cohérent car une même personne peut acheter plusieurs exemplaires d'un même ouvrage (sortie de livres, achat pour offrir ...) : 425 
df_lapage['quantity'] = (df_lapage.groupby(['id_prod', 'date', 'client_id'])['id_prod'].transform('count'))
df_lapage=df_lapage.sort_values('quantity',ascending=False)
df_clients_quantity = df_lapage.loc[df_lapage['quantity'] > 1, 'client_id'].unique()
df_clients_quantity.head()

,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client
342807,0_1630,2022-02-26,s_171121,c_1609,12.48,0,m,1980,3,B to B
86967,0_1440,2021-06-02,s_43170,c_6385,5.62,0,m,1981,3,B to B
126788,2_102,2021-07-18,s_64148,c_4958,59.14,2,m,1999,3,B to B
126321,2_102,2021-07-18,s_63848,c_4958,59.14,2,m,1999,3,B to B
86950,0_1440,2021-06-02,s_43170,c_6385,5.62,0,m,1981,3,B to B


,id_prod,date,session_id,client_id,price,categ,sex,birth,quantity,type_client
342807,0_1630,2022-02-26,s_171121,c_1609,12.48,0,m,1980,3,B to B
86967,0_1440,2021-06-02,s_43170,c_6385,5.62,0,m,1981,3,B to B
126788,2_102,2021-07-18,s_64148,c_4958,59.14,2,m,1999,3,B to B
126321,2_102,2021-07-18,s_63848,c_4958,59.14,2,m,1999,3,B to B
86950,0_1440,2021-06-02,s_43170,c_6385,5.62,0,m,1981,3,B to B


In [2190]:
len(clients_b2b)

425

In [ ]:
# Age des clients




In [2182]:
# Top 20 des clients en terme de CA généré
ca_par_client.head(10)

,client_id,price
677,c_1609,326039.89
4388,c_4958,290227.03
6337,c_6714,153918.60
2724,c_3454,114110.57
634,c_1570,5285.82
2513,c_3263,5276.87
1268,c_2140,5260.18
2108,c_2899,5214.05
7006,c_7319,5155.77
7715,c_7959,5135.75


In [2183]:
# Flop 20 des clients en terme de CA généré
ca_par_client.tail(15)

,client_id,price
8480,c_890,24.32
5868,c_6292,24.24
5828,c_6256,23.26
1972,c_2776,21.46
2563,c_3308,21.03
4408,c_4976,17.89
7798,c_8032,17.64
5354,c_5829,16.07
5453,c_5919,15.98
5589,c_6040,15.72
